In [1]:
import trino

In [2]:
conn = trino.dbapi.connect(
    host="trino",
    port=8080,
    user="jupyter",
    catalog="iceberg"
)

In [3]:
cur = conn.cursor()

In [4]:
cur.execute("SHOW TABLES in transform_db").fetchall()

[['aggtrades']]

In [7]:
cur.execute("SELECT * FROM serving_db.klines LIMIT 5")
rows = cur.fetchall()
for row in rows:
    print(row)

[1948896, '2025-08-01 00:00:00', 1754006400328945, 115764.07, 115829.46, 115308.55, 115313.01, 302.159, 1754007299467573]
[1948897, '2025-08-01 00:15:00', 1754007300010950, 115313.01, 115933.0, 115313.0, 115800.01, 450.539, 1754008199447993]
[1948898, '2025-08-01 00:30:00', 1754008200077603, 115800.0, 115800.0, 115423.87, 115517.98, 184.425, 1754009099900832]
[1948899, '2025-08-01 00:45:00', 1754009100223687, 115517.99, 115527.53, 114313.13, 115427.27, 1589.765, 1754009999974074]
[1948900, '2025-08-01 01:00:00', 1754010000363342, 115427.27, 115609.99, 114600.0, 114649.9, 681.883, 1754010899995166]


In [5]:
import pandas as pd

In [10]:
cur.execute("SELECT * FROM serving_db.klines LIMIT 10")
df = pd.DataFrame(cur.fetchall(), columns=[desc[0] for desc in cur.description])
df.head()

,group_id,group_date,open_time,open_price,high_price,low_price,close_price,volume,close_time
0,1948896,2025-08-01 00:00:00,1754006400328945,115764.07,115829.46,115308.55,115313.01,302.159,1754007299467573
1,1948897,2025-08-01 00:15:00,1754007300010950,115313.01,115933.00,115313.00,115800.01,450.539,1754008199447993
2,1948898,2025-08-01 00:30:00,1754008200077603,115800.00,115800.00,115423.87,115517.98,184.425,1754009099900832
3,1948899,2025-08-01 00:45:00,1754009100223687,115517.99,115527.53,114313.13,115427.27,1589.765,1754009999974074
4,1948900,2025-08-01 01:00:00,1754010000363342,115427.27,115609.99,114600.00,114649.90,681.883,1754010899995166


In [11]:
cur.execute("SELECT count(*) as count FROM transform_db.aggtrades")
df = pd.DataFrame(cur.fetchall(), columns=[desc[0] for desc in cur.description])
df.head()

,count
0,1314072


In [6]:
from datetime import datetime

In [14]:
cur.execute("""
CREATE TABLE IF NOT EXISTS serving_db.test_table (
    id INT,
    name VARCHAR,
    created_at TIMESTAMP
)
WITH (
    format = 'PARQUET',
    partitioning = ARRAY['created_at']
)
""")

In [ ]:
cur.execute("""
CREATE SCHEMA IF NOT EXISTS trino_db;
""")

In [42]:
cur.execute("SHOW TABLES in serving_db").fetchall()

[['klines'],
 ['klinesv2'],
 ['klinesv4'],
 ['klinesv5'],
 ['klinesv6'],
 ['test_table']]

In [24]:
# ------------------------------
# 3️⃣ Insert sample data
# ------------------------------
# Note: Trino only supports INSERT INTO VALUES (not bulk Python dataframes directly)
sample_data = [
    (1, 'Alice', datetime(2025, 9, 23, 15, 0)),
    (2, 'Bob', datetime(2025, 9, 23, 16, 0)),
    (3, 'Charlie', datetime(2025, 9, 23, 17, 0)),
]

In [25]:
for row in sample_data:
    cur.execute(
        "INSERT INTO serving_db.test_table (id, name, created_at) VALUES (?, ?, ?)",
        row
    )

In [27]:
# ------------------------------
# 4️⃣ Query table into Pandas
# ------------------------------
cur.execute("SELECT * FROM serving_db.test_table LIMIT 10")
df = pd.DataFrame(cur.fetchall(), columns=[desc[0] for desc in cur.description])
df.head(10)

,id,name,created_at
0,2,Bob,2025-09-23 16:00:00
1,3,Charlie,2025-09-23 17:00:00
2,1,Alice,2025-09-23 15:00:00
3,2,Bob,2025-09-23 16:00:00
4,1,Alice,2025-09-23 15:00:00
5,3,Charlie,2025-09-23 17:00:00
6,3,Charlie,2025-09-23 17:00:00
7,2,Bob,2025-09-23 16:00:00
8,1,Alice,2025-09-23 15:00:00


In [34]:
ctas_query = f"""
CREATE TABLE IF NOT EXISTS serving_db.klinesv6
WITH (
    format = 'PARQUET',
    location = 's3a://crypto-data-lake/serving_zone/klinesv6',
    partitioning = ARRAY['group_id']
) AS
SELECT 
    *
FROM serving_db.klines
"""

cur.execute(ctas_query)

cur.execute("SELECT * FROM serving_db.klinesv6 LIMIT 10")
df = pd.DataFrame(cur.fetchall(), columns=[desc[0] for desc in cur.description])
df.head()

CTAS table sales_report created!


,group_id,group_date,open_time,open_price,high_price,low_price,close_price,volume,close_time
0,1948918,2025-08-01 05:30:00,1754026200495126,115588.93,115754.50,115441.18,115510.70,148.939,1754027099999281
1,1948897,2025-08-01 00:15:00,1754007300010950,115313.01,115933.00,115313.00,115800.01,450.539,1754008199447993
2,1948913,2025-08-01 04:15:00,1754021700120707,115409.12,115559.88,115381.04,115526.01,124.464,1754022599962436
3,1948896,2025-08-01 00:00:00,1754006400328945,115764.07,115829.46,115308.55,115313.01,302.159,1754007299467573
4,1948908,2025-08-01 03:00:00,1754017200094588,115966.11,115990.69,115794.14,115816.00,204.915,1754018099924949


In [38]:
cur.execute("SELECT count(*) as count FROM serving_db.klinesv6").fetchall()

[[96]]

In [39]:
ctas_query = f"""
INSERT INTO serving_db.klinesv6
SELECT * FROM serving_db.klines
"""
cur.execute(ctas_query)

In [40]:
cur.execute("SELECT count(*) as count FROM serving_db.klinesv6").fetchall()

[[192]]

In [43]:
cur.execute("SHOW TABLES IN serving_db")
df = pd.DataFrame(cur.fetchall(), columns=[desc[0] for desc in cur.description])
df.head(10)

,Table
0,klines
1,klinesv2
2,klinesv4
3,klinesv5
4,klinesv6
5,test_table


In [45]:
cur.execute("select * from serving_db.klines limit 10")
df = pd.DataFrame(cur.fetchall(), columns=[desc[0] for desc in cur.description])
df.head(10)

,group_id,group_date,open_time,open_price,high_price,low_price,close_price,volume,close_time
0,1948896,2025-08-01 00:00:00,1754006400328945,115764.07,115829.46,115308.55,115313.01,302.159,1754007299467573
1,1948899,2025-08-01 00:45:00,1754009100223687,115517.99,115527.53,114313.13,115427.27,1589.765,1754009999974074
2,1948900,2025-08-01 01:00:00,1754010000363342,115427.27,115609.99,114600.00,114649.90,681.883,1754010899995166
3,1948901,2025-08-01 01:15:00,1754010900041356,114649.90,115271.13,114638.65,115190.38,448.235,1754011799896691
4,1948902,2025-08-01 01:30:00,1754011800063081,115190.37,115413.91,115000.00,115296.45,267.388,1754012699912234
5,1948903,2025-08-01 01:45:00,1754012700223622,115296.46,115407.71,115060.11,115331.86,258.534,1754013599937883
6,1948904,2025-08-01 02:00:00,1754013600005133,115328.67,115600.00,115221.07,115600.00,164.012,1754014499986628
7,1948905,2025-08-01 02:15:00,1754014500063435,115600.00,115810.71,115511.60,115619.94,168.411,1754015399984518
8,1948897,2025-08-01 00:15:00,1754007300010950,115313.01,115933.00,115313.00,115800.01,450.539,1754008199447993
9,1948898,2025-08-01 00:30:00,1754008200077603,115800.00,115800.00,115423.87,115517.98,184.425,1754009099900832


In [8]:
cur.execute("""
select 
    *,
    round((sum(close_price) over(order by group_id rows between 6 preceding and current row)) / 7, 2) as ma7
from serving_db.klines
limit 10
""")
df = pd.DataFrame(cur.fetchall(), columns=[desc[0] for desc in cur.description])
df.head(10)

,group_id,group_date,open_time,open_price,high_price,low_price,close_price,volume,close_time,ma7
0,1948896,2025-08-01 00:00:00,1754006400328945,115764.07,115829.46,115308.55,115313.01,302.159,1754007299467573,16473.29
1,1948897,2025-08-01 00:15:00,1754007300010950,115313.01,115933.00,115313.00,115800.01,450.539,1754008199447993,33016.15
2,1948898,2025-08-01 00:30:00,1754008200077603,115800.00,115800.00,115423.87,115517.98,184.425,1754009099900832,49518.71
3,1948899,2025-08-01 00:45:00,1754009100223687,115517.99,115527.53,114313.13,115427.27,1589.765,1754009999974074,66008.32
4,1948900,2025-08-01 01:00:00,1754010000363342,115427.27,115609.99,114600.00,114649.90,681.883,1754010899995166,82386.88
5,1948901,2025-08-01 01:15:00,1754010900041356,114649.90,115271.13,114638.65,115190.38,448.235,1754011799896691,98842.65
6,1948902,2025-08-01 01:30:00,1754011800063081,115190.37,115413.91,115000.00,115296.45,267.388,1754012699912234,115313.57
7,1948903,2025-08-01 01:45:00,1754012700223622,115296.46,115407.71,115060.11,115331.86,258.534,1754013599937883,115316.26
8,1948904,2025-08-01 02:00:00,1754013600005133,115328.67,115600.00,115221.07,115600.00,164.012,1754014499986628,115287.69
9,1948905,2025-08-01 02:15:00,1754014500063435,115600.00,115810.71,115511.60,115619.94,168.411,1754015399984518,115302.26


In [9]:
ctas_query = f"""
CREATE TABLE IF NOT EXISTS serving_db.ma7
WITH (
    format = 'PARQUET',
    location = 's3a://crypto-data-lake/serving_zone/ma7'
) AS
select 
    *,
    round((sum(close_price) over(order by group_id rows between 6 preceding and current row)) / 7, 2) as ma7
from serving_db.klines
"""

cur.execute(ctas_query)




,group_id,group_date,open_time,open_price,high_price,low_price,close_price,volume,close_time,ma7
0,1948896,2025-08-01 00:00:00,1754006400328945,115764.07,115829.46,115308.55,115313.01,302.159,1754007299467573,16473.29
1,1948897,2025-08-01 00:15:00,1754007300010950,115313.01,115933.00,115313.00,115800.01,450.539,1754008199447993,33016.15
2,1948898,2025-08-01 00:30:00,1754008200077603,115800.00,115800.00,115423.87,115517.98,184.425,1754009099900832,49518.71
3,1948899,2025-08-01 00:45:00,1754009100223687,115517.99,115527.53,114313.13,115427.27,1589.765,1754009999974074,66008.32
4,1948900,2025-08-01 01:00:00,1754010000363342,115427.27,115609.99,114600.00,114649.90,681.883,1754010899995166,82386.88


In [11]:
cur.execute("SELECT * FROM serving_db.ma7 offset 6 LIMIT 10")
df = pd.DataFrame(cur.fetchall(), columns=[desc[0] for desc in cur.description])
df.head(10)

,group_id,group_date,open_time,open_price,high_price,low_price,close_price,volume,close_time,ma7
0,1948899,2025-08-01 00:45:00,1754009100223687,115517.99,115527.53,114313.13,115427.27,1589.765,1754009999974074,66008.32
1,1948900,2025-08-01 01:00:00,1754010000363342,115427.27,115609.99,114600.00,114649.90,681.883,1754010899995166,82386.88
2,1948901,2025-08-01 01:15:00,1754010900041356,114649.90,115271.13,114638.65,115190.38,448.235,1754011799896691,98842.65
3,1948902,2025-08-01 01:30:00,1754011800063081,115190.37,115413.91,115000.00,115296.45,267.388,1754012699912234,115313.57
4,1948905,2025-08-01 02:15:00,1754014500063435,115600.00,115810.71,115511.60,115619.94,168.411,1754015399984518,115302.26
5,1948906,2025-08-01 02:30:00,1754015400022016,115619.95,116019.30,115572.53,115900.01,221.423,1754016299869454,115369.79
6,1948907,2025-08-01 02:45:00,1754016300091550,115900.01,116019.06,115843.42,115966.12,151.109,1754017199928964,115557.82
7,1948908,2025-08-01 03:00:00,1754017200094588,115966.11,115990.69,115794.14,115816.00,204.915,1754018099924949,115647.20
8,1948909,2025-08-01 03:15:00,1754018100126077,115816.00,116052.00,115816.00,116020.38,124.544,1754018999884991,115750.62
9,1948910,2025-08-01 03:30:00,1754019000385085,116020.39,116035.61,115828.61,115842.50,119.948,1754019899994161,115823.56


In [13]:
cur.execute("SELECT count(*) FROM serving_db.ma7").fetchall()

[[96]]